

###Esta aplicação extrairá áudio de todos os arquivos de vídeo em uma pasta do Google Drive e criará uma transcrição de alta qualidade com o sistema de reconhecimento automático de fala Whisper da OpenAI.

*Nota: Isso requer dar permissão à aplicação para se conectar ao seu drive. Apenas você terá acesso ao conteúdo do seu drive.*

Esta aplicação:
1. Conecta-se ao seu Google Drive quando você dá permissão.
2. Cria uma pasta WhisperVideo e 2 subpastas.
3. Quando você executa a aplicação, ela procurará por todos os arquivos de vídeo ou áudio na pasta principal, transcreverá e depois moverá o arquivo processado para a pasta de arquivos processados e a transcrição na pasta de textos das transcrições.

###**Para um desempenho mais rápido, configure seu ambiente de execução para "GPU"**
*Clique em "Runtime" no menu e clique em "Alterar tipo de ambiente de execução". Selecione "GPU".*

**Nota: Se você adicionar um novo arquivo após executar esta aplicação, será necessário remontar o drive na etapa 1 para torná-los pesquisáveis**


In [ ]:
# -*- coding: utf-8 -*-
"""[whisper 2026_02] Transcrever Mídias de 1 Pasta do Drive.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1aCHTj-lTDdFoXQEAvr4ni5sa906-S1px

###Esta aplicação extrairá áudio de todos os arquivos de vídeo em uma pasta do Google Drive e criará uma transcrição de alta qualidade com o sistema de reconhecimento automático de fala Whisper da OpenAI.

*Nota: Isso requer dar permissão à aplicação para se conectar ao seu drive. Apenas você terá acesso ao conteúdo do seu drive.*

Esta aplicação:
1. Conecta-se ao seu Google Drive quando você dá permissão.
2. Cria uma pasta WhisperVideo e 2 subpastas.
3. Quando você executa a aplicação, ela procurará por todos os arquivos de vídeo ou áudio na pasta principal, transcreverá e depois moverá o arquivo processado para a pasta de arquivos processados e a transcrição na pasta de textos das transcrições.

###**Para um desempenho mais rápido, configure seu ambiente de execução para "GPU"**
*Clique em "Runtime" no menu e clique em "Alterar tipo de ambiente de execução". Selecione "GPU".*

**Nota: Se você adicionar um novo arquivo após executar esta aplicação, será necessário remontar o drive na etapa 1 para torná-los pesquisáveis**
"""

# @title Licensed under the Apache License, Version 2.0 (the "License");
# Você não pode usar este arquivo exceto em conformidade com a Licença.
# Você pode obter uma cópia da Licença em
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# A menos que exigido pela lei aplicável ou acordado por escrito, o software
# distribuído sob a Licença é distribuído "COMO ESTÁ",
# SEM GARANTIAS OU CONDIÇÕES DE QUALQUER TIPO, expressas ou implícitas.
# Consulte a Licença para o texto específico que rege as permissões e
# limitações sob a Licença.

# Configuração
!pip install -U -q "openai-whisper" transformers torch accelerate

from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
import io
import os
import re
import time
import sys
import subprocess
import textwrap

# Autenticação do usuário (com cache em pickle no Drive)
from google.colab import drive, auth
import pickle
import google.auth
from google.auth.transport.requests import Request

drive.mount('/content/drive')

TOKEN_PATH = '/content/drive/MyDrive/token_colab.pickle'

creds = None
if os.path.exists(TOKEN_PATH):
    with open(TOKEN_PATH, 'rb') as token_file:
        creds = pickle.load(token_file)

if creds and creds.expired and creds.refresh_token:
    try:
        creds.refresh(Request())
        with open(TOKEN_PATH, 'wb') as token_file:
            pickle.dump(creds, token_file)
        print("🔑 Token expirado: renovado com sucesso.")
    except Exception:
        creds = None

if not creds or not getattr(creds, 'valid', False):
    auth.authenticate_user()
    creds, _ = google.auth.default()
    with open(TOKEN_PATH, 'wb') as token_file:
        pickle.dump(creds, token_file)
    print(f"🔑 Token salvo em: {TOKEN_PATH}")
else:
    print(f"🔑 Token reutilizado de: {TOKEN_PATH}")

# Analisar os argumentos
modelo_whisper = 'large-v3'  # @param ["tiny", "base", "small", "medium", "large", "large-v2", "large-v3", "pierreguillou/whisper-medium-portuguese"] {allow-input: true}

# Link da pasta no Google Drive
PASTA_DOCUMENTOS = "https://drive.google.com/drive/folders/1L-bNGHrzGnIj2Ffg97lqpCPr8zE483K6?usp=drive_link" # @param {type:"string"}

if not PASTA_DOCUMENTOS:
    raise ValueError("❌ Erro: Nenhum link da pasta do Google Drive foi fornecido. Por favor, insira um link válido.")

# Ação para áudios extraídos de vídeos
ACAO_AUDIOS = "Guardar áudios extraídos"  # @param ["Apagar áudios extraídos", "Guardar áudios extraídos"]
GUARDAR_AUDIOS = (ACAO_AUDIOS == "Guardar áudios extraídos")

# Comando para transcrição de mídias
COMANDO_TRANSCRITOR = textwrap.dedent("""# Transcrever arquivos de áudio e vídeo
Você é um Transcritor Jurídico especializado em audiências e depoimentos. Sua tarefa é transcrever fielmente os arquivos de áudio e vídeo fornecidos, respeitando a fala de cada participante (ex: Promotor, Réu, Testemunha, Juiz). Não resuma, não interprete, apenas transcreva integralmente o conteúdo falado. Identifique os interlocutores sempre que possível. O texto deve ser claro, com pontuação adequada, mantendo a fidelidade ao original. Jamais invente falas ou preencha lacunas com suposições. Se houver trechos ininteligíveis, indique como [áudio ininteligível].""")

# --- Flag e setup para modelo HuggingFace ---
IS_HF_MODEL = "/" in modelo_whisper

# --- Tag do modelo para o nome dos documentos ---
MODEL_TAG = f"[{modelo_whisper.split('/')[-1]}]"

# --- Marcadores usados no documento consolidado para identificar seções ---
MARCADOR_INICIO = "═══ TRANSCRIÇÃO: "
MARCADOR_FIM = "═══ FIM DA TRANSCRIÇÃO: "
MARCADOR_SUMARIO = "══════ SUMÁRIO ══════"


def setup_whisper():
    if IS_HF_MODEL:
        from transformers import pipeline
        import torch
        device = 0 if torch.cuda.is_available() else "cpu"
        print(f"🔧 Carregando modelo HuggingFace: {modelo_whisper}")
        print(f"   Dispositivo: {'GPU (CUDA)' if device == 0 else 'CPU'}")
        pipe = pipeline(
            task="automatic-speech-recognition",
            model=modelo_whisper,
            chunk_length_s=30,
            device=device,
            ignore_warning=True,
        )
        pipe.model.config.forced_decoder_ids = (
            pipe.tokenizer.get_decoder_prompt_ids(language="pt", task="transcribe")
        )
        return pipe
    else:
        import whisper
        model = whisper.load_model(modelo_whisper)
        return model


def extract_audio(video_path):
    """Extrai áudio de um arquivo de vídeo para WAV usando ffmpeg."""
    audio_path = video_path.rsplit(".", 1)[0] + ".wav"
    subprocess.run(
        ["ffmpeg", "-y", "-i", video_path, "-vn", "-acodec", "pcm_s16le", "-ar", "16000", "-ac", "1", audio_path],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )
    return audio_path


# Inicialização do modelo
model = setup_whisper()


def get_documents_from_drive(drive_folder_link):
    """Obtém arquivos de mídia da pasta do Google Drive e identifica documento consolidado existente."""
    print("\nIniciando leitura dos documentos na pasta do Drive...")
    MEDIA_MIME_TYPES = ['video/', 'audio/']

    try:
        folder_id = drive_folder_link.split('folders/')[-1].split('?')[0]
        print(f"ID da pasta: {folder_id}")
        service = build('drive', 'v3', cache_discovery=False, credentials=creds)

        # Listar todos os arquivos da pasta
        items = []
        page_token = None
        while True:
            response = service.files().list(
                q=f"'{folder_id}' in parents and trashed = false and mimeType != 'application/vnd.google-apps.shortcut'",
                fields="nextPageToken, files(id, name, mimeType)",
                pageSize=1000,
                pageToken=page_token
            ).execute()
            items.extend(response.get('files', []))
            page_token = response.get('nextPageToken', None)
            if not page_token:
                break

        if not items:
            print('Nenhum arquivo encontrado na pasta.')
            return {}, folder_id, None

        print(f"Encontrados {len(items)} arquivos na pasta.")

        # --- Separar arquivos de mídia e documentos existentes ---
        arquivos_midia = {}
        arquivos_ignorados = []
        doc_consolidado_id = None
        doc_consolidado_nome = None

        # Procurar documento consolidado existente (começa com o MODEL_TAG e "Transcrições de")
        prefixo_consolidado = f"{MODEL_TAG} Transcrições de"
        for item in items:
            if (item['mimeType'] == 'application/vnd.google-apps.document'
                    and item['name'].startswith(prefixo_consolidado)):
                doc_consolidado_id = item['id']
                doc_consolidado_nome = item['name']
                print(f"\n📄 Documento consolidado encontrado: '{doc_consolidado_nome}' (ID: {doc_consolidado_id})")
                break

        # Coletar arquivos de mídia
        processed_ids = set()
        for item in items:
            if item['id'] in processed_ids:
                continue
            processed_ids.add(item['id'])

            if not any(item['mimeType'].startswith(mt) for mt in MEDIA_MIME_TYPES):
                arquivos_ignorados.append((item['name'], item['mimeType']))
                continue

            arquivos_midia[item['id']] = {
                'name': item['name'],
                'mimeType': item['mimeType']
            }

        print(f"\nArquivos de mídia encontrados: {len(arquivos_midia)}")
        if arquivos_ignorados:
            print(f"Arquivos ignorados (não multimídia): {len(arquivos_ignorados)}")

        return arquivos_midia, folder_id, doc_consolidado_id

    except Exception as e:
        print(f"Erro ao acessar a pasta: {e}")
        return {}, None, None


def ler_conteudo_doc(document_id):
    """Lê o conteúdo de texto de um Google Doc existente."""
    docs_service = build('docs', 'v1', cache_discovery=False, credentials=creds)
    doc = docs_service.documents().get(documentId=document_id).execute()
    conteudo = ""
    for element in doc.get('body', {}).get('content', []):
        if 'paragraph' in element:
            for run in element['paragraph'].get('elements', []):
                if 'textRun' in run:
                    conteudo += run['textRun']['content']
    return conteudo


def extrair_arquivos_ja_transcritos(conteudo_doc):
    """Analisa o conteúdo do documento consolidado e retorna os nomes dos arquivos já transcritos."""
    ja_transcritos = set()
    # Procura pelo marcador de início de transcrição
    padrao = re.escape(MARCADOR_INICIO) + r"(.+?)\s*═══"
    matches = re.findall(padrao, conteudo_doc)
    for match in matches:
        ja_transcritos.add(match.strip())
    return ja_transcritos


def baixar_arquivo_midia(service, file_id, file_name):
    """Baixa um arquivo de mídia do Google Drive."""
    request = service.files().get_media(fileId=file_id)
    fh = io.BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        status, done = downloader.next_chunk()
    return fh.getvalue()


def obter_ou_criar_pasta_audios(folder_id):
    """Obtém ou cria a pasta 'Áudios Extraídos' dentro da pasta do Drive."""
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)
    nome_pasta = "Áudios Extraídos"
    response = drive_service.files().list(
        q=f"'{folder_id}' in parents and name='{nome_pasta}' and mimeType='application/vnd.google-apps.folder' and trashed=false",
        fields='files(id, name)'
    ).execute()
    files = response.get('files', [])
    if files:
        print(f"📁 Pasta '{nome_pasta}' já existe.")
        return files[0]['id']
    file_metadata = {
        'name': nome_pasta,
        'mimeType': 'application/vnd.google-apps.folder',
        'parents': [folder_id]
    }
    file = drive_service.files().create(body=file_metadata, fields='id').execute()
    print(f"📁 Pasta '{nome_pasta}' criada com sucesso.")
    return file.get('id')


def upload_audio_to_drive(local_path, folder_id, file_name):
    """Faz upload de um arquivo de áudio local para uma pasta do Google Drive."""
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)
    file_metadata = {
        'name': file_name,
        'parents': [folder_id]
    }
    media = MediaFileUpload(local_path, mimetype='audio/wav')
    drive_service.files().create(body=file_metadata, media_body=media, fields='id').execute()


def transcrever_arquivo(model, file_info_with_content, guardar_audio=False):
    """Transcreve um único arquivo de mídia. Retorna (transcrição, caminho_audio_extraido ou None)."""
    safe_name = file_info_with_content['name'].replace('/', '_')
    file_path = f"/tmp/{safe_name}"
    with open(file_path, "wb") as f:
        f.write(file_info_with_content['content'])

    audio_extraido_path = None
    try:
        is_video = file_info_with_content['mimeType'].startswith('video/')

        if IS_HF_MODEL:
            audio_path = extract_audio(file_path)
            result = model(audio_path)
            transcription = result["text"].strip()
            if guardar_audio and is_video:
                audio_extraido_path = audio_path
        else:
            result = model.transcribe(file_path, language="pt")
            transcription = result["text"].strip()
            if guardar_audio and is_video:
                audio_extraido_path = extract_audio(file_path)

        return (transcription if transcription else None), audio_extraido_path
    except Exception as e:
        print(f"❌ Erro ao transcrever {file_info_with_content['name']}: {e}")
        return None, None
    finally:
        # Limpar arquivo original temporário
        if os.path.exists(file_path):
            os.remove(file_path)
        # Limpar wav temporário apenas se não for guardar
        wav_path = file_path.rsplit(".", 1)[0] + ".wav"
        if not audio_extraido_path and os.path.exists(wav_path):
            os.remove(wav_path)


def criar_google_doc(folder_id, title):
    """Cria um documento Google Docs na pasta especificada."""
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)
    file_metadata = {
        'name': title,
        'mimeType': 'application/vnd.google-apps.document',
        'parents': [folder_id]
    }
    file = drive_service.files().create(body=file_metadata, fields='id').execute()
    file_id = file.get('id')
    print(f"\n📄 Documento '{title}' criado com sucesso (ID: {file_id})")
    return file_id


def obter_tamanho_doc(document_id):
    """Obtém o índice final (tamanho) do documento para append."""
    docs_service = build('docs', 'v1', cache_discovery=False, credentials=creds)
    doc = docs_service.documents().get(documentId=document_id).execute()
    # O último índice do corpo do documento
    body = doc.get('body', {})
    content = body.get('content', [])
    if content:
        last_element = content[-1]
        return last_element.get('endIndex', 1) - 1
    return 1


def inserir_conteudo_no_doc(document_id, content, index=1):
    """Insere conteúdo no documento Google Docs numa posição específica."""
    docs_service = build('docs', 'v1', cache_discovery=False, credentials=creds)
    requests = [
        {
            'insertText': {
                'location': {'index': index},
                'text': content
            }
        }
    ]
    docs_service.documents().batchUpdate(
        documentId=document_id,
        body={'requests': requests}
    ).execute()


def substituir_sumario(document_id, novo_sumario_texto):
    """Substitui o sumário no documento. Lê o doc, encontra a seção do sumário e a substitui."""
    docs_service = build('docs', 'v1', cache_discovery=False, credentials=creds)
    doc = docs_service.documents().get(documentId=document_id).execute()

    # Reconstruir o texto completo com índices
    full_text = ""
    for element in doc.get('body', {}).get('content', []):
        if 'paragraph' in element:
            for run in element['paragraph'].get('elements', []):
                if 'textRun' in run:
                    full_text += run['textRun']['content']

    # Encontrar onde o sumário começa e termina
    inicio_sumario = full_text.find(MARCADOR_SUMARIO)
    if inicio_sumario == -1:
        return  # Sem sumário para substituir

    # O sumário vai do marcador até a primeira transcrição (ou até o próximo marcador)
    inicio_primeira_transcricao = full_text.find(MARCADOR_INICIO)
    if inicio_primeira_transcricao == -1:
        fim_sumario = len(full_text)
    else:
        fim_sumario = inicio_primeira_transcricao

    # Precisamos dos índices no documento (offset +1 por causa do índice do Google Docs)
    # Nota: o índice do Google Docs começa em 1
    doc_inicio = inicio_sumario + 1
    doc_fim = fim_sumario + 1

    requests = [
        {
            'deleteContentRange': {
                'range': {
                    'startIndex': doc_inicio,
                    'endIndex': doc_fim,
                }
            }
        },
        {
            'insertText': {
                'location': {'index': doc_inicio},
                'text': novo_sumario_texto
            }
        }
    ]
    docs_service.documents().batchUpdate(
        documentId=document_id,
        body={'requests': requests}
    ).execute()


def montar_texto_sumario(lista_arquivos, ja_transcritos_set):
    """Monta o texto do sumário com a lista de arquivos e status."""
    linhas = [f"{MARCADOR_SUMARIO}\n\n"]
    for i, nome in enumerate(lista_arquivos, 1):
        status = "✅" if nome in ja_transcritos_set else "⏳"
        linhas.append(f"{i}. {status} {nome}\n")
    linhas.append(f"\nTotal: {len(lista_arquivos)} arquivo(s)\n")
    linhas.append(f"Transcritos: {len(ja_transcritos_set)} | Pendentes: {len(lista_arquivos) - len(ja_transcritos_set)}\n")
    linhas.append("\n" + "═" * 60 + "\n\n")
    return "".join(linhas)


def montar_bloco_transcricao(nome_arquivo, transcricao):
    """Monta o bloco de texto de uma transcrição individual."""
    bloco = (
        f"\n{MARCADOR_INICIO}{nome_arquivo} ═══\n"
        f"\n{transcricao}\n\n"
        f"{MARCADOR_FIM}{nome_arquivo} ═══\n"
        f"\n{'─' * 60}\n"
    )
    return bloco


# ════════════════════════════════════════════════════════════════
# FUNÇÃO PRINCIPAL
# ════════════════════════════════════════════════════════════════

def generate_transcriptions():
    # 1. Listar arquivos da pasta
    arquivos_midia, folder_id, doc_consolidado_id = get_documents_from_drive(PASTA_DOCUMENTOS)

    if not arquivos_midia:
        print("⚠️ Nenhum arquivo de áudio ou vídeo encontrado na pasta. Encerrando.")
        return None

    # Ordenar por nome para consistência
    lista_ordenada = sorted(arquivos_midia.items(), key=lambda x: x[1]['name'])
    nomes_arquivos = [info['name'] for _, info in lista_ordenada]

    # 2. Verificar se já existe documento consolidado e quais arquivos já foram transcritos
    ja_transcritos = set()
    if doc_consolidado_id:
        print("\n🔍 Lendo documento existente para verificar transcrições já feitas...")
        conteudo_existente = ler_conteudo_doc(doc_consolidado_id)
        ja_transcritos = extrair_arquivos_ja_transcritos(conteudo_existente)
        if ja_transcritos:
            print(f"\n✅ Arquivos já transcritos ({len(ja_transcritos)}):")
            for nome in ja_transcritos:
                print(f"   • {nome}")

    # 3. Determinar quais arquivos ainda precisam ser transcritos
    pendentes = [(fid, info) for fid, info in lista_ordenada if info['name'] not in ja_transcritos]

    if not pendentes:
        print("\n🎉 Todos os arquivos já foram transcritos! Nada a fazer.")
        if doc_consolidado_id:
            link = f"https://docs.google.com/document/d/{doc_consolidado_id}/edit"
            print(f"📄 Documento: {link}")
            return [link]
        return None

    print(f"\n📋 Arquivos pendentes de transcrição ({len(pendentes)}):")
    for _, info in pendentes:
        print(f"   ⏳ {info['name']}")

    # 4. Criar ou reutilizar o documento consolidado
    if not doc_consolidado_id:
        # Nome: "Transcrições de [primeiro arquivo] e outros"
        primeiro_nome = nomes_arquivos[0]
        if len(nomes_arquivos) > 1:
            doc_title = f"{MODEL_TAG} Transcrições de {primeiro_nome} e outros"
        else:
            doc_title = f"{MODEL_TAG} Transcrições de {primeiro_nome}"

        doc_consolidado_id = criar_google_doc(folder_id, doc_title)

        # Inserir sumário inicial
        sumario_texto = montar_texto_sumario(nomes_arquivos, ja_transcritos)
        inserir_conteudo_no_doc(doc_consolidado_id, sumario_texto, index=1)
        print("📑 Sumário inicial inserido no documento.")

    # 5. Transcrever arquivos pendentes e adicionar ao documento
    drive_service = build('drive', 'v3', cache_discovery=False, credentials=creds)

    # Criar pasta para áudios extraídos se configurado
    pasta_audios_id = None
    if GUARDAR_AUDIOS:
        pasta_audios_id = obter_ou_criar_pasta_audios(folder_id)

    for i, (file_id, file_info) in enumerate(pendentes):
        nome = file_info['name']
        print(f"\n{'='*60}")
        print(f"🎤 [{i+1}/{len(pendentes)}] Transcrevendo: {nome}")
        print(f"{'='*60}")

        # Baixar o arquivo
        try:
            print(f"   ⬇️  Baixando arquivo...")
            conteudo_arquivo = baixar_arquivo_midia(drive_service, file_id, nome)
        except Exception as e:
            print(f"   ❌ Erro ao baixar {nome}: {e}")
            continue

        file_info_com_conteudo = {
            'name': nome,
            'content': conteudo_arquivo,
            'mimeType': file_info['mimeType']
        }

        # Transcrever
        start_time = time.time()
        transcricao, audio_extraido_path = transcrever_arquivo(model, file_info_com_conteudo, guardar_audio=GUARDAR_AUDIOS)
        elapsed = time.time() - start_time

        if not transcricao:
            print(f"   ❌ Transcrição vazia ou com erro para {nome}. Pulando...")
            # Limpar áudio extraído residual se houver
            if audio_extraido_path and os.path.exists(audio_extraido_path):
                os.remove(audio_extraido_path)
            continue

        print(f"   ✅ Transcrição concluída em {elapsed:.1f}s ({len(transcricao)} caracteres)")

        # Upload do áudio extraído se configurado
        if audio_extraido_path and pasta_audios_id:
            try:
                nome_audio = os.path.splitext(nome)[0] + ".wav"
                upload_audio_to_drive(audio_extraido_path, pasta_audios_id, nome_audio)
                print(f"   🔊 Áudio extraído salvo no Drive: '{nome_audio}'")
            except Exception as e:
                print(f"   ⚠️ Erro ao salvar áudio extraído: {e}")
            finally:
                if os.path.exists(audio_extraido_path):
                    os.remove(audio_extraido_path)

        # Montar bloco e inserir no final do documento
        bloco = montar_bloco_transcricao(nome, transcricao)
        try:
            fim_doc = obter_tamanho_doc(doc_consolidado_id)
            inserir_conteudo_no_doc(doc_consolidado_id, bloco, index=fim_doc)
            print(f"   📝 Transcrição adicionada ao documento consolidado.")
        except Exception as e:
            print(f"   ❌ Erro ao inserir transcrição no documento: {e}")
            continue

        # Atualizar set de transcritos
        ja_transcritos.add(nome)

        # Atualizar o sumário no documento
        try:
            novo_sumario = montar_texto_sumario(nomes_arquivos, ja_transcritos)
            substituir_sumario(doc_consolidado_id, novo_sumario)
            print(f"   📑 Sumário atualizado.")
        except Exception as e:
            print(f"   ⚠️ Não foi possível atualizar o sumário: {e}")

    # 6. Resultado final
    link = f"https://docs.google.com/document/d/{doc_consolidado_id}/edit"
    print(f"\n{'='*60}")
    print(f"🎉 Processamento concluído!")
    print(f"   Transcritos: {len(ja_transcritos)}/{len(nomes_arquivos)}")
    print(f"   📄 Documento: {link}")
    print(f"{'='*60}")
    return [link]


# ════════════════════════════════════════════════════════════════
# EXECUÇÃO
# ════════════════════════════════════════════════════════════════

resultado = generate_transcriptions()
if resultado:
    print("\n✅ Todas as transcrições foram salvas no documento consolidado!")
else:
    print("\nHouve um erro ou não havia arquivos para transcrever.")

    #Limpar Memória Cache em Caso de Erro (só rodar essa célula)

import torch, gc
gc.collect()
torch.cuda.empty_cache()